# Multifractal Analysis of Heart Rate Variability Across Development
## WEA 2026 Conference Paper

This notebook implements a complete multifractal analysis pipeline for characterizing cardiac autonomic maturation through RR interval time series. The workflow combines:

- **Robust RR preprocessing** with multiple strategies (Malik filter, Lipponen-Tarvainen, age-dependent physiological ranges) and a sensitivity analysis.
- **Multifractal Detrended Fluctuation Analysis (MFDFA)** to extract the multifractal spectrum $f(\alpha)$.
- **Wavelet Transform Modulus Maxima (WTMM)** as a methodological cross-check.
- **Surrogate data tests** (shuffling and phase-randomization) to disentangle sources of multifractality.
- **Developmental comparisons** across eight age groups (neonates to adults).
- **Statistical testing** with Kruskal-Wallis, Dunn-Holm post hoc, and age-correlation analyses.
- **Publication-ready figures and tables**.

## Dataset

**RR interval time series from healthy subjects**  
PhysioNet: <https://physionet.org/content/rr-interval-healthy-subjects/1.0.0/>

## Hypothesis

The width $\Delta\alpha$ and asymmetry of the multifractal spectrum follow a developmental trajectory: narrow and symmetric in neonates, broadening and shifting during infancy and childhood, and stabilizing in adulthood. Surrogate tests will reveal whether the developmental change is dominated by long-range correlations or by the non-Gaussian distribution of increments.


## 1. Environment setup

Install dependencies. The `MFDFA` and `PyWavelets` packages are not preinstalled in Colab.


In [ ]:
!pip install -q MFDFA==0.4.3 PyWavelets nolds scikit-posthocs networkx


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import signal, stats
from scipy.interpolate import interp1d
from scipy.stats import kruskal, spearmanr
import scikit_posthocs as sp

from MFDFA import MFDFA
import pywt

np.random.seed(42)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'serif'
})

print('Environment ready.')


## 2. Paths (edit for your Colab / local environment)

If running on Colab, mount Google Drive and point `BASE_PATH` to the PhysioNet dataset folder.


In [ ]:
# --- Uncomment for Colab ---
# from google.colab import drive
# drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Paper_TDA_HRV/rr_data/rr-interval-time-series-from-healthy-subjects-1.0.0'
OUTPUT_DIR = '/content/drive/MyDrive/Paper_WEA2026_MFDFA/results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

print('BASE_PATH :', BASE_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 3. Load metadata and RR series


In [ ]:
meta_file = os.path.join(BASE_PATH, 'patient-info.csv')
df_meta = pd.read_csv(meta_file)
print('Metadata shape:', df_meta.shape)
df_meta.head()


In [ ]:
def get_file_id(filename):
    return str(int(os.path.splitext(os.path.basename(filename))[0]))

files = [
    os.path.join(BASE_PATH, f)
    for f in os.listdir(BASE_PATH)
    if f.endswith('.txt') and os.path.splitext(f)[0].isdigit()
]
print('RR files detected:', len(files))

data = []
for f in files:
    file_id = get_file_id(f)
    rr_series = []
    with open(f, 'r') as fh:
        for line in fh:
            try:
                rr_series.append(float(line.strip()))
            except Exception:
                continue
    data.append({'File': file_id, 'RR_series': rr_series})

df_rr = pd.DataFrame(data)
df_meta['File'] = df_meta['File'].astype(str)
df_rr['File'] = df_rr['File'].astype(str)

df_full = df_meta.merge(df_rr, on='File', how='inner')
print('Merged dataset shape:', df_full.shape)
df_full.head()


## 4. Developmental age groups


In [ ]:
def precise_age_group(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 0.083:
        return 'Neonates (0-1 mo)'
    elif age <= 0.42:
        return 'Early Infancy (1-5 mo)'
    elif age <= 0.92:
        return 'Late Infancy (6-11 mo)'
    elif age <= 2:
        return 'Toddlers (1-2 yr)'
    elif age <= 5:
        return 'Preschoolers (3-5 yr)'
    elif age <= 11:
        return 'School-age (6-11 yr)'
    elif age <= 17:
        return 'Adolescents (12-17 yr)'
    else:
        return 'Adults (>=18 yr)'

age_order = [
    'Neonates (0-1 mo)',
    'Early Infancy (1-5 mo)',
    'Late Infancy (6-11 mo)',
    'Toddlers (1-2 yr)',
    'Preschoolers (3-5 yr)',
    'School-age (6-11 yr)',
    'Adolescents (12-17 yr)',
    'Adults (>=18 yr)'
]

df_full['AgeGroup'] = df_full['Age (years)'].apply(precise_age_group)
print('Counts per age group:')
display(df_full['AgeGroup'].value_counts().reindex(age_order))


## 5. Robust RR preprocessing

We implement three complementary preprocessing pipelines and compare them in a sensitivity analysis.

- **P1 — Physiological range + cubic interpolation.** Age-dependent limits replace the naive `[300, 2000] ms` window that is unrealistic for neonates (resting HR ~ 120-160 bpm can reach 200+ bpm, so a lower bound of ~260 ms is needed). Artifactual beats are *interpolated* rather than removed, preserving temporal structure for downstream multifractal analysis.
- **P2 — Malik filter.** Rejects any beat deviating > 20% from the previous one and interpolates.
- **P3 — Lipponen-Tarvainen 2019** threshold-based correction (the method used in Kubios).

All three produce uniformly-sampled RR sequences; for multifractal analysis we work on the resulting `RR[n]` sequence (beat-domain) which is the standard in the literature.


In [ ]:
def age_dependent_rr_range(age_years):
    """Return (rr_min_ms, rr_max_ms) based on pediatric HR reference centiles.
    Conservative bounds derived from Fleming et al. (2011) HR centiles.
    """
    if pd.isna(age_years):
        return 300, 2000
    if age_years <= 0.25:          # 0-3 months: HR 100-205 bpm
        return 290, 600
    elif age_years <= 1:           # 3-12 months: HR 100-190 bpm
        return 315, 600
    elif age_years <= 3:           # 1-3 yr: HR 90-150 bpm
        return 400, 670
    elif age_years <= 5:           # 3-5 yr: HR 80-140 bpm
        return 430, 750
    elif age_years <= 12:          # 6-12 yr: HR 70-120 bpm
        return 500, 860
    elif age_years <= 18:          # 12-18 yr: HR 60-110 bpm
        return 545, 1000
    else:                          # Adults: HR 40-110 bpm
        return 545, 1500

def interpolate_artifacts(rr, mask_valid):
    """Replace invalid beats with cubic interpolation over the beat index.
    Preserves sequence length and temporal structure.
    """
    rr = np.asarray(rr, dtype=float).copy()
    idx = np.arange(len(rr))
    if mask_valid.sum() < 4 or mask_valid.sum() == len(rr):
        return rr, mask_valid
    # Cubic interpolation on valid points
    f = interp1d(idx[mask_valid], rr[mask_valid], kind='cubic',
                 fill_value='extrapolate', assume_sorted=True)
    rr_corrected = rr.copy()
    rr_corrected[~mask_valid] = f(idx[~mask_valid])
    return rr_corrected, mask_valid

def preprocess_P1(rr, age_years):
    """Age-dependent physiological range + cubic interpolation."""
    rr = np.asarray(rr, dtype=float)
    lo, hi = age_dependent_rr_range(age_years)
    mask = (rr >= lo) & (rr <= hi)
    rr_clean, _ = interpolate_artifacts(rr, mask)
    return rr_clean, mask

def preprocess_P2_malik(rr, age_years, threshold=0.20):
    """Malik filter: reject beats deviating > threshold from previous valid beat."""
    rr = np.asarray(rr, dtype=float)
    lo, hi = age_dependent_rr_range(age_years)
    mask_phys = (rr >= lo) & (rr <= hi)
    mask = mask_phys.copy()
    last_valid = None
    for i in range(len(rr)):
        if not mask_phys[i]:
            mask[i] = False
            continue
        if last_valid is None:
            last_valid = rr[i]
            continue
        if abs(rr[i] - last_valid) / last_valid > threshold:
            mask[i] = False
        else:
            last_valid = rr[i]
    rr_clean, _ = interpolate_artifacts(rr, mask)
    return rr_clean, mask

def preprocess_P3_lipponen(rr, age_years, alpha=5.2, c1=0.13, c2=0.17):
    """Simplified Lipponen-Tarvainen 2019 artifact correction.
    Uses median-absolute deviation of dRR and second-derivative criteria.
    """
    rr = np.asarray(rr, dtype=float)
    lo, hi = age_dependent_rr_range(age_years)
    mask = (rr >= lo) & (rr <= hi)
    if mask.sum() < 10:
        return rr.copy(), mask
    drr = np.diff(rr, prepend=rr[0])
    # Running 91-beat MAD (Lipponen & Tarvainen 2019)
    win = min(91, max(11, len(rr)//10 if len(rr)//10 % 2 == 1 else len(rr)//10 + 1))
    med_drr = pd.Series(drr).rolling(win, center=True, min_periods=1).median().values
    mad_drr = pd.Series(np.abs(drr - med_drr)).rolling(win, center=True, min_periods=1).median().values
    th1 = alpha * mad_drr / 0.6745
    outlier = np.abs(drr - med_drr) > th1
    mask = mask & (~outlier)
    rr_clean, _ = interpolate_artifacts(rr, mask)
    return rr_clean, mask

PIPELINES = {
    'P1_phys':     preprocess_P1,
    'P2_malik':    preprocess_P2_malik,
    'P3_lipponen': preprocess_P3_lipponen,
}

print('Preprocessing pipelines defined: P1_phys, P2_malik, P3_lipponen')


## 6. Quality control

A record is retained if:
- Artifact fraction (beats rejected by P1) < 8%
- Longest consecutive artifact block < 20 s (~20 beats @ 60 bpm, ~50 beats in neonates)
- After preprocessing the series has at least `MIN_LEN_BEATS` beats.


In [ ]:
MIN_LEN_BEATS = 2000   # minimum length for stable MFDFA
MAX_LEN_BEATS = 20000  # truncate very long recordings for comparability

def qc_record(rr, age_years, art_frac_thr=0.08, max_block_beats=30):
    rr = np.asarray(rr, dtype=float)
    lo, hi = age_dependent_rr_range(age_years)
    mask = (rr >= lo) & (rr <= hi)
    art_frac = 1.0 - mask.mean() if len(rr) else 1.0
    # Longest invalid block
    invalid = (~mask).astype(int)
    max_block = 0
    count = 0
    for v in invalid:
        if v == 1:
            count += 1
            max_block = max(max_block, count)
        else:
            count = 0
    exclude = (art_frac > art_frac_thr) or (max_block > max_block_beats) or (len(rr) < MIN_LEN_BEATS)
    return {
        'art_frac': art_frac,
        'max_block_beats': max_block,
        'n_beats_raw': len(rr),
        'exclude': exclude
    }

qc_rows = []
for _, row in df_full.iterrows():
    res = qc_record(row['RR_series'], row['Age (years)'])
    qc_rows.append(res)
qc_df = pd.DataFrame(qc_rows)
df_full_qc = pd.concat([df_full.reset_index(drop=True), qc_df], axis=1)

print('Retained:', (df_full_qc['exclude']==False).sum(), '/', len(df_full_qc))
summary_qc = (df_full_qc.groupby('AgeGroup', observed=False)
              .apply(lambda g: pd.Series({
                  'N_total': len(g),
                  'N_retained': (g['exclude']==False).sum(),
                  'Retained_%': 100*(g['exclude']==False).mean(),
                  'Median_artifact_frac': g['art_frac'].median(),
                  'Median_max_block': g['max_block_beats'].median(),
                  'Median_n_beats': g['n_beats_raw'].median()
              }))).reset_index()
summary_qc['AgeGroup'] = pd.Categorical(summary_qc['AgeGroup'], categories=age_order, ordered=True)
summary_qc = summary_qc.sort_values('AgeGroup').reset_index(drop=True)
summary_qc.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'qc_summary.csv'), index=False)
display(summary_qc)


## 7. MFDFA implementation

Multifractal Detrended Fluctuation Analysis following Kantelhardt et al. (2002).

Given a time series $x(t)$ of length $N$, we:
1. Build the profile $Y(i) = \sum_{k=1}^{i}[x(k) - \langle x \rangle]$.
2. Divide it into $N_s = \lfloor N/s \rfloor$ non-overlapping windows of length $s$.
3. Fit a polynomial (order $m$, default 2) in each window and compute the residual variance $F^2(s, \nu)$.
4. Generalize to q-th order fluctuations:
$$F_q(s) = \left[\frac{1}{N_s}\sum_{\nu=1}^{N_s} [F^2(s,\nu)]^{q/2}\right]^{1/q}$$
5. The scaling $F_q(s) \sim s^{h(q)}$ yields the generalized Hurst exponent $h(q)$.
6. The multifractal spectrum is obtained via the Legendre transform: $\tau(q) = qh(q) - 1$, $\alpha = d\tau/dq$, $f(\alpha) = q\alpha - \tau(q)$.

We extract: $\Delta\alpha = \alpha_{\max} - \alpha_{\min}$ (spectrum width), $\alpha_0$ (spectrum peak), $\Delta f$, asymmetry $A = (\alpha_{\max} - \alpha_0)/(\alpha_0 - \alpha_{\min})$, and $h(2)$ (standard Hurst).


In [ ]:
def compute_mfdfa(rr, q_list=None, scale_min=16, scale_max=None, n_scales=20, order=2):
    """Run MFDFA and return h(q), tau(q), alpha, f(alpha) and scalar descriptors."""
    rr = np.asarray(rr, dtype=float)
    N = len(rr)
    if q_list is None:
        q_list = np.linspace(-5, 5, 21)
        q_list = q_list[np.abs(q_list) > 0.05]  # avoid q = 0
    if scale_max is None:
        scale_max = N // 8
    scales = np.unique(np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int))
    scales = scales[scales >= scale_min]

    lag, dfa = MFDFA(rr, lag=scales, q=q_list, order=order)

    # Generalized Hurst exponent h(q) from log-log slope of F_q(s) vs s
    h_q = np.zeros(len(q_list))
    for i, q in enumerate(q_list):
        valid = (dfa[:, i] > 0) & np.isfinite(dfa[:, i])
        if valid.sum() < 4:
            h_q[i] = np.nan
            continue
        slope, _ = np.polyfit(np.log(lag[valid]), np.log(dfa[valid, i]), 1)
        h_q[i] = slope

    tau = q_list * h_q - 1
    # Legendre transform via finite differences
    alpha = np.gradient(tau, q_list)
    f_alpha = q_list * alpha - tau

    # Clean spectrum (remove NaN/Inf)
    valid = np.isfinite(alpha) & np.isfinite(f_alpha)
    alpha_v, f_v, q_v = alpha[valid], f_alpha[valid], q_list[valid]

    if len(alpha_v) < 3:
        return {
            'q': q_list, 'h_q': h_q, 'tau': tau, 'alpha': alpha, 'f_alpha': f_alpha,
            'scales': lag,
            'delta_alpha': np.nan, 'alpha_0': np.nan, 'delta_f': np.nan,
            'asymmetry': np.nan, 'hurst_h2': np.nan,
            'h_q_range': np.nan
        }

    alpha_min, alpha_max = np.min(alpha_v), np.max(alpha_v)
    delta_alpha = alpha_max - alpha_min

    # alpha_0 = location of f(alpha) maximum
    alpha_0 = alpha_v[np.argmax(f_v)]
    delta_f = np.max(f_v) - np.min(f_v)

    # Asymmetry (right/left spectrum width ratio)
    left = alpha_0 - alpha_min
    right = alpha_max - alpha_0
    asymmetry = (right - left) / (right + left) if (right + left) > 0 else np.nan

    # h(2) = standard Hurst
    idx_q2 = np.argmin(np.abs(q_list - 2.0))
    hurst_h2 = h_q[idx_q2]

    # Range of h(q)
    h_q_range = np.nanmax(h_q) - np.nanmin(h_q)

    return {
        'q': q_list, 'h_q': h_q, 'tau': tau, 'alpha': alpha, 'f_alpha': f_alpha,
        'scales': lag,
        'delta_alpha': delta_alpha,
        'alpha_0': alpha_0,
        'delta_f': delta_f,
        'asymmetry': asymmetry,
        'hurst_h2': hurst_h2,
        'h_q_range': h_q_range
    }

print('MFDFA routine ready.')


## 8. Surrogate data: shuffled and phase-randomized

To identify the *source* of multifractality we generate two types of surrogates:

- **Shuffled surrogate (SS)**: random permutation of the series. Preserves the marginal PDF but destroys all temporal correlations. If the original $\Delta\alpha$ is significantly larger than that of SS, long-range correlations contribute to multifractality.
- **Phase-randomized surrogate (PRS / FT)**: same power spectrum but randomized Fourier phases. Preserves linear correlations but destroys non-linearities and non-Gaussianity. If the original $\Delta\alpha$ is larger than that of PRS, non-Gaussian distribution / non-linearity contribute.


In [ ]:
def shuffle_surrogate(x, rng=None):
    rng = rng or np.random.default_rng()
    y = x.copy()
    rng.shuffle(y)
    return y

def phase_randomized_surrogate(x, rng=None):
    """FT surrogate: randomize Fourier phases while preserving amplitudes."""
    rng = rng or np.random.default_rng()
    n = len(x)
    X = np.fft.rfft(x - np.mean(x))
    amps = np.abs(X)
    # Random phases for interior frequencies; preserve DC and Nyquist real
    phases = rng.uniform(0, 2*np.pi, size=len(X))
    phases[0] = 0
    if n % 2 == 0:
        phases[-1] = 0
    X_surr = amps * np.exp(1j*phases)
    y = np.fft.irfft(X_surr, n=n) + np.mean(x)
    return y

def mfdfa_surrogate_test(rr, n_surr=20, seed=42):
    """Return delta_alpha for original and for n_surr SS and PRS surrogates."""
    rng = np.random.default_rng(seed)
    orig = compute_mfdfa(rr)
    ss_da, prs_da = [], []
    for _ in range(n_surr):
        ss = shuffle_surrogate(rr, rng)
        prs = phase_randomized_surrogate(rr, rng)
        ss_da.append(compute_mfdfa(ss)['delta_alpha'])
        prs_da.append(compute_mfdfa(prs)['delta_alpha'])
    return {
        'delta_alpha_orig': orig['delta_alpha'],
        'delta_alpha_ss_mean': np.nanmean(ss_da),
        'delta_alpha_ss_std': np.nanstd(ss_da),
        'delta_alpha_prs_mean': np.nanmean(prs_da),
        'delta_alpha_prs_std': np.nanstd(prs_da),
    }

print('Surrogate routines ready.')


## 9. Wavelet Transform Modulus Maxima (WTMM) cross-check

A lightweight WTMM implementation based on continuous wavelet transform with Mexican-hat wavelet. We estimate $h(q)$ from the scaling of the partition function $Z(q, a)$ across scales $a$.


In [ ]:
def wtmm_spectrum(x, scales=None, q_list=None):
    x = np.asarray(x, dtype=float)
    n = len(x)
    if scales is None:
        scales = np.unique(np.logspace(np.log10(4), np.log10(n//16), 16).astype(int))
        scales = scales[scales >= 2]
    if q_list is None:
        q_list = np.linspace(-5, 5, 21)
        q_list = q_list[np.abs(q_list) > 0.05]

    # Continuous wavelet transform with Mexican-hat (mexh)
    coefs, _ = pywt.cwt(x, scales, 'mexh')
    # Track modulus maxima along the scale axis
    Z = np.zeros((len(q_list), len(scales)))
    for si, s in enumerate(scales):
        coef_s = np.abs(coefs[si])
        # Local maxima
        local_max_idx = signal.argrelextrema(coef_s, np.greater)[0]
        if len(local_max_idx) < 3:
            Z[:, si] = np.nan
            continue
        maxima = coef_s[local_max_idx]
        for qi, q in enumerate(q_list):
            Z[qi, si] = np.sum(maxima ** q)

    # Scaling exponents tau(q)
    tau = np.zeros(len(q_list))
    for qi in range(len(q_list)):
        valid = np.isfinite(Z[qi]) & (Z[qi] > 0)
        if valid.sum() < 4:
            tau[qi] = np.nan
            continue
        slope, _ = np.polyfit(np.log(scales[valid]), np.log(Z[qi, valid]), 1)
        tau[qi] = slope

    alpha = np.gradient(tau, q_list)
    f_alpha = q_list * alpha - tau

    valid = np.isfinite(alpha) & np.isfinite(f_alpha)
    alpha_v, f_v = alpha[valid], f_alpha[valid]
    if len(alpha_v) < 3:
        return {'delta_alpha_wtmm': np.nan, 'alpha_0_wtmm': np.nan,
                'asymmetry_wtmm': np.nan, 'q': q_list, 'tau': tau,
                'alpha': alpha, 'f_alpha': f_alpha}
    alpha_min, alpha_max = alpha_v.min(), alpha_v.max()
    delta_alpha = alpha_max - alpha_min
    alpha_0 = alpha_v[np.argmax(f_v)]
    left = alpha_0 - alpha_min
    right = alpha_max - alpha_0
    asym = (right - left) / (right + left) if (right + left) > 0 else np.nan
    return {
        'delta_alpha_wtmm': delta_alpha,
        'alpha_0_wtmm': alpha_0,
        'asymmetry_wtmm': asym,
        'q': q_list, 'tau': tau, 'alpha': alpha, 'f_alpha': f_alpha
    }

print('WTMM routine ready.')


## 10. Feature extraction: master loop

For each retained subject we compute:
- MFDFA descriptors on the series preprocessed with P1 (main pipeline).
- Surrogate descriptors (reduced `n_surr` for efficiency — set to 20 for the paper).
- WTMM descriptors (cross-check).
- Classical HRV (mean RR, SDNN, RMSSD, pNN50) as controls.

⚠️ This cell is the most time-consuming. On Colab it typically takes **30-60 min** depending on dataset size. For quick iteration set `QUICK_MODE = True`.


In [ ]:
QUICK_MODE = False   # set True to reduce n_surr and subsample subjects for debugging
N_SURR = 5 if QUICK_MODE else 20

def classical_hrv(rr):
    rr = np.asarray(rr, dtype=float)
    drr = np.diff(rr)
    return {
        'mean_rr': rr.mean(),
        'mean_hr': 60000.0 / rr.mean(),
        'sdnn': rr.std(ddof=1),
        'rmssd': np.sqrt(np.mean(drr**2)),
        'pnn50': np.mean(np.abs(drr) > 50)
    }

df_analysis = df_full_qc[df_full_qc['exclude']==False].copy().reset_index(drop=True)
if QUICK_MODE:
    df_analysis = df_analysis.groupby('AgeGroup', observed=False).head(3).reset_index(drop=True)
print('Subjects to analyze:', len(df_analysis))

results = []
for i, row in df_analysis.iterrows():
    rr_raw = row['RR_series']
    age = row['Age (years)']
    # Main pipeline: P1
    rr_clean, _ = preprocess_P1(rr_raw, age)
    # Truncate
    if len(rr_clean) > MAX_LEN_BEATS:
        rr_clean = rr_clean[:MAX_LEN_BEATS]
    if len(rr_clean) < MIN_LEN_BEATS:
        continue

    try:
        mf = compute_mfdfa(rr_clean)
        surr = mfdfa_surrogate_test(rr_clean, n_surr=N_SURR, seed=int(row['File']) % 10000)
        wt = wtmm_spectrum(rr_clean)
        hrv = classical_hrv(rr_clean)
    except Exception as e:
        print(f'Subject {row["File"]} failed: {e}')
        continue

    results.append({
        'File': row['File'],
        'Age': age,
        'Gender': row.get('Gender', np.nan),
        'AgeGroup': row['AgeGroup'],
        'n_beats': len(rr_clean),
        # MFDFA
        'delta_alpha': mf['delta_alpha'],
        'alpha_0': mf['alpha_0'],
        'delta_f': mf['delta_f'],
        'asymmetry': mf['asymmetry'],
        'hurst_h2': mf['hurst_h2'],
        'h_q_range': mf['h_q_range'],
        # Surrogates
        'da_ss_mean': surr['delta_alpha_ss_mean'],
        'da_ss_std': surr['delta_alpha_ss_std'],
        'da_prs_mean': surr['delta_alpha_prs_mean'],
        'da_prs_std': surr['delta_alpha_prs_std'],
        'ratio_orig_ss': mf['delta_alpha'] / surr['delta_alpha_ss_mean'] if surr['delta_alpha_ss_mean'] else np.nan,
        'ratio_orig_prs': mf['delta_alpha'] / surr['delta_alpha_prs_mean'] if surr['delta_alpha_prs_mean'] else np.nan,
        # WTMM
        'delta_alpha_wtmm': wt['delta_alpha_wtmm'],
        'alpha_0_wtmm': wt['alpha_0_wtmm'],
        'asymmetry_wtmm': wt['asymmetry_wtmm'],
        # HRV
        **hrv
    })
    if (i+1) % 10 == 0:
        print(f'  Processed {i+1}/{len(df_analysis)}')

df_feats = pd.DataFrame(results)
df_feats['AgeGroup'] = pd.Categorical(df_feats['AgeGroup'], categories=age_order, ordered=True)
df_feats = df_feats.sort_values(['AgeGroup', 'Age']).reset_index(drop=True)
df_feats.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'features_mfdfa.csv'), index=False)
print('Feature table shape:', df_feats.shape)
df_feats.head()


## 11. Sensitivity analysis of preprocessing

We recompute $\Delta\alpha$ using P2 (Malik) and P3 (Lipponen) for a random subset of subjects, and check consistency of the group-level effect.


In [ ]:
SENS_SUBSET = 40 if not QUICK_MODE else 10
sens_df = df_analysis.sample(min(SENS_SUBSET, len(df_analysis)), random_state=0).reset_index(drop=True)

sens_rows = []
for _, row in sens_df.iterrows():
    rr_raw = row['RR_series']
    age = row['Age (years)']
    out = {'File': row['File'], 'Age': age, 'AgeGroup': row['AgeGroup']}
    for name, func in PIPELINES.items():
        try:
            rr_clean, _ = func(rr_raw, age)
            if len(rr_clean) > MAX_LEN_BEATS:
                rr_clean = rr_clean[:MAX_LEN_BEATS]
            if len(rr_clean) < MIN_LEN_BEATS:
                out[f'delta_alpha_{name}'] = np.nan
                continue
            mf = compute_mfdfa(rr_clean)
            out[f'delta_alpha_{name}'] = mf['delta_alpha']
        except Exception:
            out[f'delta_alpha_{name}'] = np.nan
    sens_rows.append(out)

df_sens = pd.DataFrame(sens_rows)
df_sens.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'sensitivity_preprocessing.csv'), index=False)

# Pairwise correlations
print('Pairwise Spearman correlations between pipelines (delta_alpha):')
cols = [c for c in df_sens.columns if c.startswith('delta_alpha_')]
print(df_sens[cols].corr(method='spearman').round(3))
df_sens.head()


In [ ]:
# Sensitivity plot: scatter of pipelines vs each other
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
pairs = [('delta_alpha_P1_phys', 'delta_alpha_P2_malik'),
         ('delta_alpha_P1_phys', 'delta_alpha_P3_lipponen'),
         ('delta_alpha_P2_malik', 'delta_alpha_P3_lipponen')]
for ax, (x, y) in zip(axes, pairs):
    sub = df_sens[[x, y]].dropna()
    ax.scatter(sub[x], sub[y], s=30, alpha=0.7, edgecolor='k', linewidth=0.4)
    mn = min(sub[x].min(), sub[y].min())
    mx = max(sub[x].max(), sub[y].max())
    ax.plot([mn, mx], [mn, mx], 'k--', alpha=0.5, label='y=x')
    r = sub[x].corr(sub[y], method='spearman')
    ax.set_xlabel(x.replace('delta_alpha_', r'$\Delta\alpha$ '))
    ax.set_ylabel(y.replace('delta_alpha_', r'$\Delta\alpha$ '))
    ax.set_title(f'Spearman $\\rho$ = {r:.2f}')
    ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig_sensitivity_preprocessing.png'))
plt.show()


## 12. Statistical analysis


In [ ]:
metrics_main = ['delta_alpha', 'alpha_0', 'asymmetry', 'hurst_h2',
                'h_q_range', 'delta_alpha_wtmm', 'asymmetry_wtmm',
                'ratio_orig_ss', 'ratio_orig_prs',
                'sdnn', 'rmssd', 'pnn50']

metric_labels = {
    'delta_alpha': r'$\Delta\alpha$ (MFDFA width)',
    'alpha_0': r'$\alpha_0$ (spectrum peak)',
    'asymmetry': 'Asymmetry $A$',
    'hurst_h2': 'Hurst $h(2)$',
    'h_q_range': r'range $h(q)$',
    'delta_alpha_wtmm': r'$\Delta\alpha$ (WTMM)',
    'asymmetry_wtmm': 'Asymmetry (WTMM)',
    'ratio_orig_ss': r'$\Delta\alpha_{orig}/\Delta\alpha_{SS}$',
    'ratio_orig_prs': r'$\Delta\alpha_{orig}/\Delta\alpha_{PRS}$',
    'sdnn': 'SDNN (ms)',
    'rmssd': 'RMSSD (ms)',
    'pnn50': 'pNN50'
}

# Kruskal-Wallis
kw_rows = []
dunn_results = {}
for m in metrics_main:
    samples = [df_feats.loc[df_feats['AgeGroup']==g, m].dropna().values for g in age_order]
    samples = [s for s in samples if len(s) > 0]
    if len(samples) >= 2:
        H, p = kruskal(*samples)
    else:
        H, p = np.nan, np.nan
    # Correlation with age (continuous)
    sub = df_feats[['Age', m]].dropna()
    if len(sub) >= 10:
        rho, p_rho = spearmanr(sub['Age'], sub[m])
    else:
        rho, p_rho = np.nan, np.nan
    kw_rows.append({'metric': m, 'H': H, 'p_kw': p, 'spearman_rho': rho, 'p_spearman': p_rho})
    # Dunn post hoc
    sub2 = df_feats[['AgeGroup', m]].dropna()
    if sub2['AgeGroup'].nunique() >= 2:
        dunn = sp.posthoc_dunn(sub2, val_col=m, group_col='AgeGroup', p_adjust='holm')
        dunn = dunn.reindex(index=age_order, columns=age_order)
        dunn_results[m] = dunn

kw_df = pd.DataFrame(kw_rows).sort_values('p_kw')
kw_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'kruskal_spearman.csv'), index=False)
display(kw_df)


## 13. Descriptive table (median [IQR])


In [ ]:
desc_metrics = ['delta_alpha', 'alpha_0', 'asymmetry', 'hurst_h2',
                'ratio_orig_ss', 'ratio_orig_prs']

table_desc = pd.DataFrame()
for m in desc_metrics:
    temp = (df_feats.groupby('AgeGroup', observed=False)[m]
            .apply(lambda x: f'{np.median(x):.3f} [{np.percentile(x,25):.3f}-{np.percentile(x,75):.3f}]'
                   if len(x.dropna()) else 'NA'))
    table_desc[m] = temp

table_desc = table_desc.loc[age_order]
table_desc.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'descriptive_table.csv'))
with open(os.path.join(OUTPUT_DIR, 'tables', 'descriptive_table.tex'), 'w') as f:
    f.write(table_desc.to_latex(escape=False))
display(table_desc)


## 14. Figure 1 — Representative multifractal spectra per age group


In [ ]:
# Pick one representative subject per group (closest to group median delta_alpha)
reps = []
for g in age_order:
    sub = df_feats[df_feats['AgeGroup']==g].dropna(subset=['delta_alpha'])
    if len(sub) == 0: continue
    med = sub['delta_alpha'].median()
    pick = sub.iloc[(sub['delta_alpha'] - med).abs().argsort()[:1]]
    reps.append((g, pick.iloc[0]['File']))

fig, ax = plt.subplots(1, 1, figsize=(8, 5.5))
colors = sns.color_palette('viridis', len(reps))

for (g, fid), col in zip(reps, colors):
    row = df_analysis[df_analysis['File']==fid].iloc[0]
    rr_clean, _ = preprocess_P1(row['RR_series'], row['Age (years)'])
    if len(rr_clean) > MAX_LEN_BEATS:
        rr_clean = rr_clean[:MAX_LEN_BEATS]
    mf = compute_mfdfa(rr_clean)
    valid = np.isfinite(mf['alpha']) & np.isfinite(mf['f_alpha'])
    ax.plot(mf['alpha'][valid], mf['f_alpha'][valid],
            'o-', color=col, label=g, markersize=4, linewidth=1.5, alpha=0.85)

ax.set_xlabel(r'$\alpha$ (Hölder exponent)')
ax.set_ylabel(r'$f(\alpha)$')
ax.set_title('Representative multifractal spectra across development')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.45), ncol=2, fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig1_spectra_representative.png'))
plt.show()


## 15. Figure 2 — Boxplots of multifractal descriptors across age groups


In [ ]:
metrics_violin = ['delta_alpha', 'alpha_0', 'asymmetry', 'hurst_h2']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for idx, m in enumerate(metrics_violin):
    ax = axes[idx]
    sns.violinplot(x='AgeGroup', y=m, data=df_feats, order=age_order,
                   inner=None, cut=0, palette='viridis', ax=ax)
    sns.boxplot(x='AgeGroup', y=m, data=df_feats, order=age_order,
                width=0.2, boxprops={'facecolor':'white','zorder':2},
                showfliers=False, whiskerprops={'linewidth':1.5}, ax=ax)
    sns.stripplot(x='AgeGroup', y=m, data=df_feats, order=age_order,
                  color='black', size=2.5, jitter=0.2, alpha=0.4, ax=ax)
    ax.text(-0.08, 1.04, panel_labels[idx], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='bottom', ha='right')
    ax.set_title(metric_labels[m], fontsize=13, weight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(metric_labels[m])
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    # Put rotation anchor right-aligned via xticklabels
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig2_boxplots.png'))
plt.show()


## 16. Figure 3 — Dunn-Holm heatmaps for $\Delta\alpha$ and $\alpha_0$


In [ ]:
def significance_marker(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''

sel = ['delta_alpha', 'alpha_0', 'asymmetry', 'hurst_h2']
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for idx, feature in enumerate(sel):
    ax = axes[idx]
    if feature not in dunn_results:
        ax.axis('off'); continue
    df_dunn = dunn_results[feature].loc[age_order, age_order]
    annot = df_dunn.applymap(lambda x: f'{x:.3f}\n{significance_marker(x)}' if pd.notna(x) else '')
    sns.heatmap(df_dunn, annot=annot, fmt='', cmap=sns.light_palette('navy', as_cmap=True),
                cbar_kws={'label': 'Holm-adj. p'}, linewidths=0.4, vmin=0, vmax=1,
                annot_kws={'size': 8}, ax=ax)
    ax.text(-0.08, 1.04, panel_labels[idx], transform=ax.transAxes,
            fontsize=13, fontweight='bold', va='bottom', ha='right')
    ax.set_title(metric_labels[feature], fontsize=12, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig3_dunn_heatmaps.png'))
plt.show()


## 17. Figure 4 — Surrogate analysis: origin of multifractality

For each age group we compare $\Delta\alpha_{orig}$, $\Delta\alpha_{SS}$ and $\Delta\alpha_{PRS}$. A large gap orig-vs-SS implies correlation-driven multifractality; a large gap orig-vs-PRS implies non-Gaussian / non-linear contribution.


In [ ]:
surr_long = df_feats.melt(id_vars=['AgeGroup'],
                          value_vars=['delta_alpha','da_ss_mean','da_prs_mean'],
                          var_name='type', value_name='delta_alpha')
surr_long['type'] = surr_long['type'].map({
    'delta_alpha': 'Original',
    'da_ss_mean': 'Shuffled',
    'da_prs_mean': 'Phase-rand.'
})

fig, ax = plt.subplots(1, 1, figsize=(11, 5.5))
sns.boxplot(x='AgeGroup', y='delta_alpha', hue='type',
            data=surr_long, order=age_order,
            palette={'Original':'#264653', 'Shuffled':'#e76f51', 'Phase-rand.':'#2a9d8f'},
            showfliers=False, ax=ax)
ax.set_ylabel(r'$\Delta\alpha$')
ax.set_xlabel('')
ax.set_title('Origin of multifractality: original vs surrogates', weight='bold')
ax.tick_params(axis='x', rotation=35)
for tick in ax.get_xticklabels():
    tick.set_ha('right')
ax.legend(title='', loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig4_surrogates.png'))
plt.show()


## 18. Figure 5 — MFDFA vs WTMM cross-check


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
sub = df_feats[['delta_alpha','delta_alpha_wtmm','AgeGroup']].dropna()
sns.scatterplot(data=sub, x='delta_alpha', y='delta_alpha_wtmm',
                hue='AgeGroup', hue_order=age_order, palette='viridis',
                ax=axes[0], s=40, edgecolor='k', linewidth=0.3)
mn = min(sub['delta_alpha'].min(), sub['delta_alpha_wtmm'].min())
mx = max(sub['delta_alpha'].max(), sub['delta_alpha_wtmm'].max())
axes[0].plot([mn,mx],[mn,mx],'k--', alpha=0.5)
rho = sub['delta_alpha'].corr(sub['delta_alpha_wtmm'], method='spearman')
axes[0].set_xlabel(r'$\Delta\alpha$ (MFDFA)')
axes[0].set_ylabel(r'$\Delta\alpha$ (WTMM)')
axes[0].set_title(f'Cross-check (Spearman $\\rho$ = {rho:.2f})')
axes[0].legend(fontsize=8, loc='best')

sub2 = df_feats[['asymmetry','asymmetry_wtmm','AgeGroup']].dropna()
sns.scatterplot(data=sub2, x='asymmetry', y='asymmetry_wtmm',
                hue='AgeGroup', hue_order=age_order, palette='viridis',
                ax=axes[1], s=40, edgecolor='k', linewidth=0.3, legend=False)
axes[1].axhline(0, color='gray', ls=':', alpha=0.5)
axes[1].axvline(0, color='gray', ls=':', alpha=0.5)
rho2 = sub2['asymmetry'].corr(sub2['asymmetry_wtmm'], method='spearman')
axes[1].set_xlabel('Asymmetry (MFDFA)')
axes[1].set_ylabel('Asymmetry (WTMM)')
axes[1].set_title(f'Spearman $\\rho$ = {rho2:.2f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig5_mfdfa_vs_wtmm.png'))
plt.show()


## 19. Figure 6 — Developmental trajectory of $\Delta\alpha$ vs chronological age


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5.2))
sub = df_feats[['Age','delta_alpha','AgeGroup']].dropna()
sns.scatterplot(data=sub, x='Age', y='delta_alpha', hue='AgeGroup',
                hue_order=age_order, palette='viridis',
                s=40, edgecolor='k', linewidth=0.3, ax=ax)
# LOWESS fit
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    sm = lowess(sub['delta_alpha'], sub['Age'], frac=0.35, return_sorted=True)
    ax.plot(sm[:,0], sm[:,1], 'k-', linewidth=2.5, alpha=0.85, label='LOWESS')
except Exception:
    pass
ax.set_xlabel('Age (years, log-scaled)')
ax.set_ylabel(r'$\Delta\alpha$')
ax.set_xscale('symlog', linthresh=0.1)
ax.set_title('Developmental trajectory of multifractal spectrum width', weight='bold')
ax.legend(fontsize=8, loc='best', ncol=2)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig6_trajectory.png'))
plt.show()


## 20. Classification (age-group discrimination) with multifractal features

Baseline model: Random Forest on multifractal features vs classical HRV.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

feat_sets = {
    'MFDFA': ['delta_alpha','alpha_0','asymmetry','hurst_h2','h_q_range',
              'ratio_orig_ss','ratio_orig_prs'],
    'WTMM': ['delta_alpha_wtmm','alpha_0_wtmm','asymmetry_wtmm'],
    'Classical HRV': ['mean_rr','sdnn','rmssd','pnn50'],
    'MFDFA + WTMM': ['delta_alpha','alpha_0','asymmetry','hurst_h2','h_q_range',
                     'delta_alpha_wtmm','alpha_0_wtmm','asymmetry_wtmm'],
    'MFDFA + HRV': ['delta_alpha','alpha_0','asymmetry','hurst_h2','h_q_range',
                    'mean_rr','sdnn','rmssd','pnn50'],
    'All': ['delta_alpha','alpha_0','asymmetry','hurst_h2','h_q_range',
            'delta_alpha_wtmm','alpha_0_wtmm','asymmetry_wtmm',
            'mean_rr','sdnn','rmssd','pnn50'],
}

y = df_feats['AgeGroup'].astype(str).values
cv_rows = []
for name, cols in feat_sets.items():
    data = df_feats[cols + ['AgeGroup']].dropna()
    if data['AgeGroup'].value_counts().min() < 2:
        continue
    X = data[cols].values
    yy = data['AgeGroup'].astype(str).values
    pipe = Pipeline([('sc', StandardScaler()),
                     ('rf', RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    acc = cross_val_score(pipe, X, yy, cv=cv, scoring='accuracy')
    f1  = cross_val_score(pipe, X, yy, cv=cv, scoring='f1_macro')
    cv_rows.append({'feature_set': name, 'n_features': len(cols),
                    'accuracy_mean': acc.mean(), 'accuracy_std': acc.std(),
                    'f1macro_mean': f1.mean(), 'f1macro_std': f1.std()})

cv_df = pd.DataFrame(cv_rows).sort_values('f1macro_mean', ascending=False)
cv_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'classification_cv.csv'), index=False)
display(cv_df)


In [ ]:
# Bar plot of classification performance
fig, ax = plt.subplots(1, 1, figsize=(8.5, 4.5))
cv_plot = cv_df.sort_values('f1macro_mean')
ax.barh(cv_plot['feature_set'], cv_plot['f1macro_mean'],
        xerr=cv_plot['f1macro_std'], color='#264653', alpha=0.85, edgecolor='k')
ax.set_xlabel('Macro-F1 (5-fold CV)')
ax.set_title('Age-group classification performance', weight='bold')
ax.set_xlim(0, 1)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fig7_classification.png'))
plt.show()


## 21. Feature importance (SHAP) — optional, slow


In [ ]:
RUN_SHAP = False  # set True if you want SHAP plots
if RUN_SHAP:
    !pip install -q shap
    import shap
    cols = feat_sets['All']
    data = df_feats[cols + ['AgeGroup']].dropna()
    X = data[cols].values
    yy = data['AgeGroup'].astype(str).values
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    rf = RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1).fit(Xs, yy)
    explainer = shap.TreeExplainer(rf)
    shap_vals = explainer.shap_values(Xs)
    shap.summary_plot(shap_vals, Xs, feature_names=cols, show=True)
else:
    print('SHAP disabled. Set RUN_SHAP = True to compute feature importance plots.')


## 22. Save all results and package for paper


In [ ]:
print('\n=== SUMMARY OF OUTPUTS ===')
for folder in ['tables', 'figures']:
    p = os.path.join(OUTPUT_DIR, folder)
    files = sorted(os.listdir(p)) if os.path.exists(p) else []
    print(f'\n[{folder}]')
    for f in files:
        size_kb = os.path.getsize(os.path.join(p, f)) / 1024
        print(f'  {f}  ({size_kb:.1f} KB)')
print('\nAll results saved under:', OUTPUT_DIR)


## 23. Key results cheatsheet for the paper

Generate a concise text block summarizing the main numbers to quote in the manuscript.


In [ ]:
def fmt_median_iqr(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0: return 'NA'
    return f'{np.median(x):.3f} [{np.percentile(x,25):.3f}-{np.percentile(x,75):.3f}]'

report = []
report.append('MULTIFRACTAL HRV ACROSS DEVELOPMENT — KEY RESULTS')
report.append('=' * 60)
report.append(f'N subjects analyzed: {len(df_feats)}')
report.append(f'Subjects per group:')
for g in age_order:
    n = (df_feats["AgeGroup"] == g).sum()
    report.append(f'  {g}: n = {n}')
report.append('')
report.append('Delta_alpha (MFDFA, median [IQR]):')
for g in age_order:
    vals = df_feats.loc[df_feats["AgeGroup"]==g, "delta_alpha"].values
    report.append(f'  {g}: {fmt_median_iqr(vals)}')
report.append('')
report.append('Kruskal-Wallis:')
for _, r in kw_df.iterrows():
    report.append(f'  {r["metric"]:20s}  H = {r["H"]:.2f}  p = {r["p_kw"]:.2e}  '
                  f'rho_age = {r["spearman_rho"]:.2f}  p_rho = {r["p_spearman"]:.2e}')
report.append('')
report.append('Classification (macro-F1, 5-fold CV):')
for _, r in cv_df.iterrows():
    report.append(f'  {r["feature_set"]:18s}  F1 = {r["f1macro_mean"]:.3f} +/- {r["f1macro_std"]:.3f}')

report_text = '\n'.join(report)
print(report_text)
with open(os.path.join(OUTPUT_DIR, 'key_results.txt'), 'w') as f:
    f.write(report_text)
print('\nSaved to', os.path.join(OUTPUT_DIR, 'key_results.txt'))


## 24. Outline for the WEA 2026 paper

**Title (tentative):** *Developmental trajectory of cardiac multifractality: MFDFA and surrogate analysis of RR intervals from neonates to adults.*

**Target:** WEA 2026 (IEEE-style conference proceeding, ~6–8 pages).

**Suggested structure:**

1. **Introduction** — Cardiac autonomic maturation, HRV as a maturation marker, limits of linear HRV, rationale for multifractal analysis; our previous HVG/TDA work motivates broader non-linear characterization.
2. **Dataset and preprocessing** — PhysioNet healthy subjects dataset, eight developmental groups, age-dependent physiological bounds, cubic interpolation, QC criteria. Mention sensitivity analysis (P1/P2/P3).
3. **Methods** — MFDFA formulation, surrogate data (shuffling, phase randomization), WTMM cross-check, classical HRV baseline, Kruskal-Wallis + Dunn-Holm + Spearman, Random Forest classification.
4. **Results** — Figure 1 (representative spectra), Figure 2 (violin plots of $\Delta\alpha,\alpha_0,$ asymmetry, Hurst), Figure 4 (surrogate analysis), Figure 6 (age trajectory), Table: descriptive + Kruskal + Spearman, classification table (feature set comparison).
5. **Discussion** — Physiological interpretation: $\Delta\alpha$ growth with age as increasing complexity of autonomic regulation; surrogate decomposition identifies whether correlations or non-Gaussianity drive the change; comparison with prior HVG/TDA results; limitations (cross-sectional, artifact sensitivity, truncation).
6. **Conclusion** — MFDFA complements HVG/TDA for developmental characterization; multifractal width is a candidate maturation index; future work: longitudinal cohorts, clinical applicability (prematurity, congenital disease).

**Figures packaged:** `fig1_spectra_representative.png`, `fig2_boxplots.png`, `fig3_dunn_heatmaps.png`, `fig4_surrogates.png`, `fig5_mfdfa_vs_wtmm.png`, `fig6_trajectory.png`, `fig7_classification.png`, `fig_sensitivity_preprocessing.png`.

**Tables packaged:** `qc_summary.csv`, `descriptive_table.csv/.tex`, `kruskal_spearman.csv`, `classification_cv.csv`, `sensitivity_preprocessing.csv`.
